NOTE FOR LATER: Had to download from https://www.redfin.com/news/data-center/downloads/ and upload to volume

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:

from pyspark.sql import SparkSession
import src.utils.helpers



In [0]:
%sql
DROP TABLE IF EXISTS bronze_dev.redfin.housing_market_tracker_zipcode;
DROP TABLE IF EXISTS bronze_dev.redfin.housing_market_tracker_county;
DROP TABLE IF EXISTS bronze_dev.redfin.housing_market_tracker_neighborhood;

In [0]:

from dataclasses import dataclass, field

# Use a dataclass for validation
@dataclass
class BronzePull:
    file_path: str
    table_name: str
    natural_key: list[str]
    file_format: str = "csv"
    options: dict = field(default_factory=dict)

data_pulls = []

for directory in ["housing_market","price_drops","delistings_relistings"]:

    for grain in [
        ["counties","county"],
        ["zips","zipcode"],
        ["neighborhoods","neightborhood"]
    ]:
        
        pull = BronzePull(
            file_path=f"s3://redfin-public-data/redfin_data_center/{directory}/monthly/all_{grain[0]}.csv",
            table_name=f"bronze_dev.redfin.{directory}_{grain[1]}",
            natural_key=["PERIOD_BEGIN", "PERIOD_END", "REGION_NAME"],
        )
        # print(f"{directory}, {grain}")
        print(pull.file_path)
        print(pull.table_name)
        data_pulls.append(pull)

# Files only available at the "metro area" level

data_pulls.extend([
    # For "available metros"
    BronzePull(
        file_path="s3://redfin-public-data/redfin_data_center/investors/by_metro/all_metros.csv",
        table_name="bronze_dev.redfin.investor_metro",
        natural_key=["PERIOD_BEGIN", "PERIOD_END", "REGION_NAME"],
    )
])



import time

for pull in data_pulls:
    start = time.time()

    src.utils.helpers.bronze_read_prep_upsert(
        file_path=pull.file_path,
        file_format=pull.file_format,
        table_name=pull.table_name,
        natural_key=pull.natural_key,
        options=pull.options,
        spark=spark,
    )

    elapsed = time.time() - start

    print(
        f"Completed {pull.table_name} "
        f"in {elapsed:.2f} seconds"
    )

In [0]:
%sql
SELECT * FROM bronze_dev.information_schema.tables where table_schema = 'redfin';

In [0]:
%sql

-- DROP TABLE bronze_dev.redfin.housing_market_tracker_neighborhood;
-- DROP TABLE  bronze_dev.redfin.housing_market_tracker_county;
-- DROP TABLE  bronze_dev.redfin.housing_market_tracker_zipcode;
-- DROP TABLE  bronze_dev.redfin.price_drops_zipcode;

In [0]:
%sql
SELECT * FROM bronze_dev.redfin.housing_market_tracker_zipcode

In [0]:
df1 = spark.read.table("bronze_dev.redfin.housing_market_tracker_zipcode")
df2 = spark.read.table("bronze_dev.redfin.housing_market_monthly_all_zips")
cols1 = set(df1.columns)
cols2 = set(df2.columns)

only_in_df1 = cols1 - cols2
only_in_df2 = cols2 - cols1
common_cols  = cols1 & cols2

print("Only in df1:", only_in_df1)
print("Only in df2:", only_in_df2)
print("Common:", common_cols)
